# SS3 Part B — STADIOEquities customer activation prediction

**Final chosen method:** Random Forest classifier with a leak-free preprocessing pipeline.  
**Target:** `activation_outcome` — whether a client activated by making their first successful deposit, as labelled by the client.  
**Unit of analysis:** one registered client. **Prediction time:** at registration.  

The supplied archive also contains a separate dormancy target. This notebook intentionally trains **one** model (activation), because the two targets have different eligible populations and permissible predictors. All random seeds are fixed to 42.

> The source README describes these records as **synthetic** and warns that many current-state and post-deposit fields leak the activation label. This notebook uses only fields available when a client registered.

## 1. Dataset loading (2 marks)

Load the two necessary CSV tables directly from  `data/table1_client_account.csv` (registration-time predictors) and `table9_outcome_target.csv` (labels only). The original files each contain **3,109 rows**, but **nine duplicate client rows** are removed, leaving **3,100 unique clients**. The binary classes are **activated (True, 1)** and **not activated (False, 0)**. The following code prints actual counts, proportions, and the majority-to-minority imbalance ratio rather than relying on manually entered estimates.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
import joblib

RANDOM_STATE = 42
def read_from_archive(filename):
    file = 'data/' + filename
    return pd.read_csv(file)

accounts_raw = read_from_archive('table1_client_account.csv')
labels_raw = read_from_archive('table9_outcome_target.csv')
print('Raw account rows:', len(accounts_raw))
print('Raw outcome rows:', len(labels_raw))
for name, data in [('accounts', accounts_raw), ('outcomes', labels_raw)]:
    conflicts = data.groupby('client_id').size().gt(1).sum()
    print(f'{name}: {conflicts} client IDs repeated')

assert labels_raw.groupby('client_id')['activation_outcome'].nunique().max() == 1, 'Conflicting target labels: investigate duplicates'
accounts = accounts_raw.drop_duplicates(subset='client_id', keep='first').copy()
labels = labels_raw.drop_duplicates(subset='client_id', keep='first').copy()
df = accounts.merge(labels[['client_id','activation_outcome']], on='client_id', how='inner', validate='one_to_one')
assert len(df) == df.client_id.nunique()
assert df.activation_outcome.notna().all()
y_all = df['activation_outcome'].astype(int)
counts = y_all.value_counts().reindex([0,1], fill_value=0)
print(f'Unique usable clients: {len(df):,}')
print(f'Not activated (0): {counts[0]:,} ({counts[0]/len(df):.1%})')
print(f'Activated (1): {counts[1]:,} ({counts[1]/len(df):.1%})')
print(f'Imbalance ratio (majority:minority): {counts.max()/counts.min():.2f}:1')
display(df[['client_id', 'registration_date', 'activation_outcome']].head())

Raw account rows: 3109
Raw outcome rows: 3109
accounts: 9 client IDs repeated
outcomes: 9 client IDs repeated
Unique usable clients: 3,100
Not activated (0): 1,567 (50.5%)
Activated (1): 1,533 (49.5%)
Imbalance ratio (majority:minority): 1.02:1


,client_id,registration_date,activation_outcome
0,C102938,2023-03-24 07:22:48,True
1,C102939,2023-03-23 07:01:16,False
2,C102943,2023-04-13 18:58:00,True
3,C102944,2022-11-08 22:39:28,False
4,C102946,2023-05-30 17:09:31,False
